In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori,association_rules
import pandas as pd

<h1 style="color:blue;"> Toy Example </h1>

In [ ]:
dataset = [['Lait', 'Oignon', 'Noix', 'Haricots Rouges', 'Oeufs', 'Yaourt'],
           ['Aneth', 'Oignon', 'Noix', 'Haricots Rouges', 'Oeufs', 'Yaourt'],
           ['Lait', 'Pomme', 'Haricots Rouges', 'Oeufs'],
           ['Lait', 'Maïs', 'Haricots Rouges', 'Yaourt'],
           ['Maïs', 'Oignon', 'Haricots Rouges', 'Glace', 'Oeufs']]

transactions = TransactionEncoder()
transactions_array = transactions.fit(dataset).transform(dataset)
df_train = pd.DataFrame(transactions_array, columns=transactions.columns_)
df_train.head(3)



### Recherche de motifs fréquents

In [ ]:
frequent_itemsets_train = apriori(df_train, min_support=0.6, use_colnames=True)
frequent_itemsets_train

In [ ]:
frequent_itemsets_train[frequent_itemsets_train['itemsets'].astype(str).str.contains("Lait")]

### Recherche des règles d'association

In [ ]:
training_rules = association_rules(frequent_itemsets_train, metric="confidence", min_threshold=0.7)
training_rules.sort_values(by=['confidence','lift'],ascending=False).iloc[:,:7]

<h1 style="color:blue;"> Market Kaggle Data Example </h1>

In [ ]:
commandes=pd.read_csv('TP/TP-20260112/order_products__prior.csv',sep=',')
commandes.head(5)

In [ ]:
commandes.shape

In [ ]:
produits=pd.read_csv('TP/TP-20260112/products.csv',sep=',')
produits.head(5)

In [ ]:
produits.shape

In [ ]:
frequent_itemsets_train

In [ ]:
training_rules

## Préparation des données

### Sélection des commandes les plus importantes

In [ ]:
commandes_count = commandes.groupby('order_id').size().reset_index(name='nb_produits')

nb_top_commandes = 10000
commandes_importantes = commandes_count.sort_values('nb_produits', ascending=False).head(nb_top_commandes)

print(f"Nombre total de commandes: {len(commandes_count)}")
print(f"Top {nb_top_commandes} commandes sélectionnées")
print(f"Nombre de produits min: {commandes_importantes['nb_produits'].min()}")
print(f"Nombre de produits max: {commandes_importantes['nb_produits'].max()}")
print(f"Nombre moyen de produits: {commandes_importantes['nb_produits'].mean():.2f}")

commandes_filtrees = commandes[commandes['order_id'].isin(commandes_importantes['order_id'])]

print(f"\nNombre de lignes avant filtrage: {len(commandes)}")
print(f"Nombre de lignes après filtrage: {len(commandes_filtrees)}")

print("\nTop 5 des commandes:")
commandes_importantes.head()

### Sélection des produits les plus achetés

In [ ]:
produits_count = commandes_filtrees.groupby('product_id').size().reset_index(name='nb_achats')

nb_top_produits = 1000
produits_populaires = produits_count.sort_values('nb_achats', ascending=False).head(nb_top_produits)

print(f"Nombre total de produits différents: {len(produits_count)}")
print(f"Nombre d'achats min dans le top: {produits_populaires['nb_achats'].min()}")
print(f"Nombre d'achats max dans le top: {produits_populaires['nb_achats'].max()}")
print(f"Nombre moyen d'achats: {produits_populaires['nb_achats'].mean():.2f}")

commandes_finales = commandes_filtrees[commandes_filtrees['product_id'].isin(produits_populaires['product_id'])]

print(f"\nNombre de lignes avant filtrage: {len(commandes_filtrees)}")
print(f"Nombre de lignes après filtrage: {len(commandes_finales)}")

top_10_produits = produits_populaires.head(10)
top_10_avec_nom = top_10_produits.merge(produits[['product_id', 'product_name']], on='product_id')
print("\nTop 10 des produits les plus achetés:")
top_10_avec_nom